In [1]:
import networkx as nx
import itertools
from math import comb
import matplotlib.pyplot as plt
from typing import List, Tuple, Set

In [2]:
def hamming_distance(a: int, b: int) -> int:
    """Distancia de Hamming entre dos enteros (popcount del XOR)."""
    return (a ^ b).bit_count()

def build_hamming_graph(n: int, d: int) -> nx.Graph:
    """
    Construye el grafo de Hamming H(n, d):
    - Vértices: números enteros de 0 a 2^n - 1 (representan palabras binarias)
    - Arista entre u y v si hamming_distance(u, v) >= d
    """
    num_vertices = 1 << n
    G = nx.Graph()
    G.add_nodes_from(range(num_vertices))
    for u in range(num_vertices):
        for v in range(u + 1, num_vertices):
            if hamming_distance(u, v) >= d:
                G.add_edge(u, v)
    return G


In [3]:
def bron_kerbosch_max_cliques(G: nx.Graph):
    """
    Implementación del algoritmo de Bron–Kerbosch (versión con pivote)
    que encuentra todos los cliques maximales en un grafo no dirigido.
    Adaptado del artículo original "Algorithm 457: Finding All Cliques of an Undirected Graph".
    Retorna una lista de cliques (cada clique es un conjunto de vértices).
    """
    # Convertir los vértices a enteros 0..N-1 (ya lo son)
    vertices = list(G.nodes())
    # Precalcular vecinos como lista de conjuntos para acceso rápido
    neighbors = {v: set(G.neighbors(v)) for v in vertices}
    N = len(vertices)
    # Orden inicial: todos los vértices en "candidates", "not" vacío
    all_vertices = vertices[:]  # lista de todos los vértices en orden
    compsub = []  # R, clique en construcción
    cliques = []  # almacenará los cliques maximales encontrados

    def extend(old, ne, ce):
        """
        old: lista de vértices (primero 'not' (0..ne-1), luego 'candidates' (ne..ce-1))
        ne: número de vértices en not (0 <= ne <= ce)
        ce: número total de elementos en old (len(old))
        """
        nonlocal cliques
        # Paso 1: elegir punto fijo (fixp) con mínimo número de disconexiones
        minnod = ce
        fixp = None
        s = -1
        nod = 0  # indicador si el punto fijo se tomó de candidates (1) o not (0)
        i = 0
        while i < ce and minnod != 0:
            p = old[i]
            count = 0
            # Contar disconexiones con el resto de candidatos (desde ne hasta ce-1)
            j = ne
            pos = -1
            while j < ce and count <= minnod:
                if p not in neighbors[old[j]]:  # disconexión
                    count += 1
                    pos = j
                j += 1
            if count < minnod:
                fixp = p
                minnod = count
                if i < ne:
                    s = pos
                else:
                    s = i
                    nod = 1
            i += 1

        # Bucle principal de backtracking
        # nod iterará desde minnod + nod hasta 1
        for _ in range(minnod + nod, 0, -1):
            # Intercambiar el candidato seleccionado (old[s]) con old[ne]
            p = old[s]
            old[s] = old[ne]
            sel = old[ne]
            old[ne] = p

            # Construir nuevo conjunto 'not' (new) y 'candidates' (newcand)
            new = [0] * ce  # preasignamos tamaño máximo
            newne = 0
            # Copiar vértices de 'not' que son vecinos de sel
            for i in range(ne):
                if sel in neighbors[old[i]]:
                    new[newne] = old[i]
                    newne += 1
            newce = newne
            # Copiar vértices de 'candidates' (desde ne+1 hasta ce-1) que son vecinos de sel
            # Nota: el índice ne ya contiene sel, que fue movido; lo saltamos
            for i in range(ne + 1, ce):
                if sel in neighbors[old[i]]:
                    new[newce] = old[i]
                    newce += 1

            compsub.append(sel)

            if newce == 0:
                # Se encontró un clique maximal
                cliques.append(compsub.copy())
            else:
                if newne < newce:
                    # Llamada recursiva con el nuevo conjunto
                    # new[:newce] contiene (not + candidates)
                    extend(new, newne, newce)

            compsub.pop()
            # Mover sel al conjunto 'not' para futuras iteraciones
            ne += 1

            # Si aún quedan candidatos por procesar (nod > 1), seleccionar el siguiente
            # candidato desconectado del punto fijo
            if nod > 1:
                # Buscar siguiente candidato (pos > s) que esté desconectado de fixp
                s = ne
                while s < ce and fixp in neighbors[old[s]]:
                    s += 1
                if s >= ce:
                    break
                # El siguiente candidato ya está en old[s], listo para el siguiente ciclo
            else:
                # Solo un candidato, terminar
                break

    # Iniciar llamada con todos los vértices en candidates, not vacío
    extend(all_vertices, 0, N)
    return cliques


In [6]:
G= build_hamming_graph(8,4)
cliques = bron_kerbosch_max_cliques(G)
cliques

[[0, 15, 51, 60, 85, 90, 102, 105, 170, 204, 240, 255]]

In [4]:
def main():
    # Parámetros de prueba
    n = 8      # longitud de las palabras
    d = 4      # distancia mínima requerida
    print(f"Construyendo grafo de Hamming H({n},{d})...")
    G = build_hamming_graph(n, d)
    print(f"Vértices: {G.number_of_nodes()}, Aristas: {G.number_of_edges()}")

    print("\n=== Cliques maximales encontrados por nuestra implementación ===")
    our_cliques = bron_kerbosch_max_cliques(G)
    print(f"Número de cliques maximales: {len(our_cliques)}")
    max_size = max(len(c) for c in our_cliques) if our_cliques else 0
    print(f"Tamaño del clique máximo (A({n},{d})): {max_size}")
    # Mostrar primeros 5 cliques como ejemplo
    print("Ejemplo de cliques (primeros 5):")
    for i, clique in enumerate(our_cliques[:5]):
        # Convertir enteros a representación binaria para mejor visualización
        bin_repr = [format(v, f'0{n}b') for v in clique]
        print(f"  {i+1}: {bin_repr}")

    # Comparación con networkx.find_cliques (Bron–Kerbosch implementado en C)
    print("\n=== Comparación con networkx.find_cliques ===")
    nx_cliques = list(nx.find_cliques(G))
    print(f"networkx encontró {len(nx_cliques)} cliques maximales.")
    nx_max_size = max(len(c) for c in nx_cliques) if nx_cliques else 0
    print(f"Tamaño del clique máximo según networkx: {nx_max_size}")

    # Verificar que nuestros cliques coinciden (como conjuntos)
    our_sets = [set(c) for c in our_cliques]
    nx_sets = [set(c) for c in nx_cliques]
    if set(frozenset(s) for s in our_sets) == set(frozenset(s) for s in nx_sets):
        print("¡Los conjuntos de cliques maximales coinciden perfectamente!")
    else:
        print("Advertencia: los conjuntos difieren. Revisar implementación.")

if __name__ == "__main__":
    main()

Construyendo grafo de Hamming H(8,4)...
Vértices: 256, Aristas: 20864

=== Cliques maximales encontrados por nuestra implementación ===
Número de cliques maximales: 1
Tamaño del clique máximo (A(8,4)): 12
Ejemplo de cliques (primeros 5):
  1: ['00000000', '00001111', '00110011', '00111100', '01010101', '01011010', '01100110', '01101001', '10101010', '11001100', '11110000', '11111111']

=== Comparación con networkx.find_cliques ===


KeyboardInterrupt: 

In [ ]:
def bron_kerbosch_version2(graph: nx.Graph) -> List[Set[int]]:
    """
    Implementación del algoritmo de Bron–Kerbosch versión 2 (con pivote)
    para encontrar todas las cliques maximales.
    Sigue la lógica del artículo "Algorithm 457" (Comm. ACM, 1973).
    """
    # Convertir el grafo a una matriz de adyacencia booleana para acceso rápido
    nodes = list(graph.nodes())
    index_of = {node: i for i, node in enumerate(nodes)}
    n_nodes = len(nodes)
    adj = [[False]*n_nodes for _ in range(n_nodes)]
    for u, v in graph.edges():
        i, j = index_of[u], index_of[v]
        adj[i][j] = adj[j][i] = True
    # Para simplificar, trabajamos con índices enteros 0..n_nodes-1
    # La función recursiva interna usará listas de enteros (índices)
    
    all_cliques = []   # almacenará las cliques encontradas (como conjuntos de nodos originales)
    
    # Algoritmo recursivo: extiende una clique parcial (compsub)
    # old: array de enteros ordenado [not | candidates] (ne = tamaño de not, ce = tamaño total)
    # ne, ce: índices que separan not y candidates en el arreglo old[0:ne] es not, old[ne:ce] es candidates
    def extend(old: List[int], ne: int, ce: int):
        # old tiene longitud ce, los primeros ne son "not", los siguientes ce-ne son "candidates"
        # Se trabaja in-place, pero creamos un nuevo arreglo new para la llamada recursiva.
        
        # Determinar el punto fijo (fixp) y el candidato con mínimo número de desconexiones
        # (versión 2 del artículo)
        minnod = ce
        fixp = -1
        s = -1           # posición del candidato que se usará como pivote
        nod = 0          # bandera: 1 si fixp viene de candidates
        i = 0
        while i < ce and minnod != 0:
            p = old[i]
            # Contar cuántos candidatos (en la parte candidates) NO son adyacentes a p
            cnt = 0
            j = ne
            pos = -1
            while j < ce and cnt <= minnod:
                if not adj[p][old[j]]:
                    cnt += 1
                    pos = j   # posición de ese candidato desconectado
                j += 1
            if cnt < minnod:
                fixp = p
                minnod = cnt
                if i < ne:
                    # fixp viene de not
                    s = pos
                else:
                    # fixp viene de candidates
                    s = i
                    nod = 1
            i += 1
        
        # Ciclo de backtracking: se repite para cada candidato elegido
        # nod = minnod + nod (en el artículo se itera desde minnod+nod hasta 1)
        # Aquí implementamos la lógica de selección del candidato a mover
        for _ in range(minnod + nod, 0, -1):
            # Seleccionar el candidato en posición s, intercambiarlo con old[ne] (el primero de candidates)
            p = old[s]
            # Intercambio
            old[s], old[ne] = old[ne], p
            sel = old[ne]
            # Construir nuevos conjuntos new (candidatos y not) basados en sel
            new = [0] * ce
            newne = 0
            # Primero los que estaban en not y son adyacentes a sel
            for i in range(ne):
                if adj[sel][old[i]]:
                    new[newne] = old[i]
                    newne += 1
            # Luego los que estaban en candidates (old[ne+1:ce]) y son adyacentes a sel
            newce = newne
            for i in range(ne+1, ce):
                if adj[sel][old[i]]:
                    new[newne] = old[i]
                    newne += 1
            # newce es el tamaño de la nueva sección candidates (new[newce:newne])
            # En el algoritmo original: newce = newne (antes de añadir candidates)
            # Pero luego se añaden los candidatos, entonces newne es el total, y newce es el inicio de candidates en new
            # En nuestro new, los primeros newce son los nuevos "not", los siguientes son candidates.
            # Notar que newce es el número de elementos que pasaron de not (adyacentes). Los otros son los nuevos candidates.
            # Ahora se añade sel a compsub
            compsub.append(sel)
            if newne == newce:   # es decir, no hay candidatos nuevos
                # Clique maximal encontrada
                clique_nodes = [nodes[v] for v in compsub]
                all_cliques.append(set(clique_nodes))
            else:
                # Llamada recursiva
                extend(new, newce, newne)
            # Quitar sel de compsub
            compsub.pop()
            # Mover sel a not (incrementar ne)
            ne += 1
            # Si todavía quedan candidatos por procesar (nod > 1 en el artículo)
            if _ > 1:
                # Buscar el siguiente candidato que NO está conectado a fixp
                # (para reducir ramas)
                s = ne
                # Avanzar hasta encontrar un candidato desconectado de fixp
                while s < ce and adj[fixp][old[s]]:
                    s += 1
                if s >= ce:
                    break   # no hay más candidatos que cumplan la condición
        # fin del ciclo for
    
    # Inicialización: ALL contiene todos los nodos (índices)
    all_indices = list(range(n_nodes))
    compsub = []    # pila para la clique actual
    # Llamada inicial: old = all_indices, ne = 0, ce = n_nodes
    extend(all_indices, 0, n_nodes)
    return all_cliques

In [3]:
def bron_kerbosch_version2c(graph: nx.Graph) -> List[Set[int]]:
    """
    Implementación del algoritmo de Bron-Kerbosch versión 2 (con pivote)
    para encontrar todas las cliques maximales en un grafo no dirigido.

    Sigue la lógica del artículo "Algorithm 457" (Comm. ACM, 1973).

    Estrategia central:
      - Se mantiene un arreglo `window` que divide en dos secciones:
          window[0 : num_excluded]          → nodos "excluidos" (ya procesados)
          window[num_excluded : window_size] → nodos "candidatos" (aún por explorar)
      - En cada paso se elige un pivote (el nodo con menos candidatos no-adyacentes)
        para reducir el número de ramas recursivas.
      - `current_clique` es una pila compartida que representa la clique en construcción.

    Complejidad: O(3^(n/3)) en el peor caso, pero el pivote lo acelera en la práctica.
    """

    # ── 1. Construir matriz de adyacencia booleana ──────────────────────────────
    # Permite verificar adyacencia en O(1), lo que acelera el conteo del pivote.

    all_nodes = list(graph.nodes())
    node_to_index = {node: idx for idx, node in enumerate(all_nodes)}
    total_nodes = len(all_nodes)

    # adj[i][j] == True  ↔  existe arista entre el nodo i y el nodo j
    adj = [[False] * total_nodes for _ in range(total_nodes)]
    for u, v in graph.edges():
        i, j = node_to_index[u], node_to_index[v]
        adj[i][j] = adj[j][i] = True   # grafo no dirigido: la relación es simétrica

    # ── 2. Resultado acumulado y pila de la clique en construcción ──────────────
    found_cliques: List[Set] = []
    current_clique: List[int] = []   # índices de los nodos en la clique parcial actual

    # ── 3. Función recursiva principal ─────────────────────────────────────────
    def extend(window: List[int], num_excluded: int, window_size: int) -> None:
        """
        Extiende la clique parcial `current_clique` usando los nodos en `window`.

        Parámetros
        ----------
        window       : lista de índices de nodos. Tiene dos secciones:
                         [0 : num_excluded]    → excluidos (no pueden ampliar la clique)
                         [num_excluded : window_size] → candidatos (podrían ampliarla)
        num_excluded : separador entre excluidos y candidatos dentro de `window`.
        window_size  : longitud efectiva de `window` (puede ser < len(window)).
        """

        # ── 3a. Elegir el pivote ────────────────────────────────────────────────
        # El pivote es el nodo (excluido o candidato) con el menor número de
        # candidatos que NO son adyacentes a él.  Esto minimiza las ramas del árbol.

        min_disconnected = window_size   # cota superior inicial (peor caso)
        pivot_node = -1
        pivot_pos_in_window = -1         # posición del primer candidato no-adyacente al pivote
        pivot_is_candidate = False       # True si el pivote viene de la sección candidatos

        for i in range(window_size):
            if min_disconnected == 0:
                break   # ya no se puede mejorar; salir pronto

            node = window[i]

            # Contar candidatos no adyacentes a `node`
            disconnected_count = 0
            last_disconnected_pos = -1

            for j in range(num_excluded, window_size):
                if not adj[node][window[j]]:
                    disconnected_count += 1
                    last_disconnected_pos = j
                    if disconnected_count > min_disconnected:
                        break   # ya superó el mínimo; no hay ganancia

            if disconnected_count < min_disconnected:
                pivot_node = node
                min_disconnected = disconnected_count
                pivot_pos_in_window = last_disconnected_pos

                if i < num_excluded:
                    # El pivote viene de la sección "excluidos":
                    # usamos la posición del candidato desconectado directamente.
                    pass
                else:
                    # El pivote viene de la sección "candidatos":
                    # su propia posición (i) es el primer candidato a probar.
                    pivot_pos_in_window = i
                    pivot_is_candidate = True

        # ── 3b. Ciclo de backtracking ───────────────────────────────────────────
        # Se itera una vez por cada candidato no-adyacente al pivote
        # (más una iteración extra si el propio pivote es candidato).
        num_iterations = min_disconnected + (1 if pivot_is_candidate else 0)

        for iteration in range(num_iterations, 0, -1):

            # Seleccionar el candidato a añadir en esta iteración:
            # intercambiarlo con el primer candidato del window (posición num_excluded).
            selected_node = window[pivot_pos_in_window]
            window[pivot_pos_in_window], window[num_excluded] = (
                window[num_excluded],
                selected_node,
            )
            selected_node = window[num_excluded]   # confirmar tras el swap

            # ── Construir el sub-window para la llamada recursiva ───────────────
            # Solo entran los nodos adyacentes a `selected_node`.
            sub_window: List[int] = []
            sub_num_excluded = 0

            # Nodos excluidos adyacentes a selected_node → siguen siendo excluidos
            for i in range(num_excluded):
                if adj[selected_node][window[i]]:
                    sub_window.append(window[i])
                    sub_num_excluded += 1

            # Candidatos adyacentes a selected_node → pasan a ser candidatos del sub-window
            for i in range(num_excluded + 1, window_size):
                if adj[selected_node][window[i]]:
                    sub_window.append(window[i])

            sub_window_size = len(sub_window)

            # ── Añadir selected_node a la clique en construcción ────────────────
            current_clique.append(selected_node)

            if sub_window_size == sub_num_excluded:
                # No hay candidatos en el sub-window: clique maximal encontrada.
                clique_nodes = {all_nodes[v] for v in current_clique}
                found_cliques.append(clique_nodes)
            else:
                # Aún hay candidatos: continuar expandiendo.
                extend(sub_window, sub_num_excluded, sub_window_size)

            # ── Retroceder: quitar selected_node de la clique ───────────────────
            current_clique.pop()

            # Mover selected_node a la sección "excluidos" para futuras iteraciones
            num_excluded += 1

            # Buscar el siguiente candidato no adyacente al pivote
            if iteration > 1:
                pivot_pos_in_window = num_excluded
                while (
                    pivot_pos_in_window < window_size
                    and adj[pivot_node][window[pivot_pos_in_window]]
                ):
                    pivot_pos_in_window += 1

                if pivot_pos_in_window >= window_size:
                    break   # no quedan candidatos válidos

    # ── 4. Llamada inicial ──────────────────────────────────────────────────────
    # Al inicio: ningún nodo está excluido (num_excluded = 0),
    # todos son candidatos.
    initial_window = list(range(total_nodes))
    extend(initial_window, num_excluded=0, window_size=total_nodes)

    return found_cliques

In [1]:
# ============== Ejemplo de uso ==============
if __name__ == "__main__":
    n = 6  # longitud de las palabras
    d = 4  # distancia mínima deseada
    print(f"Construyendo grafo de Hamming H({n},{d})...")
    G = build_hamming_graph(n, d)
    print(f"Vértices: {G.number_of_nodes()}, Aristas: {G.number_of_edges()}")
    
    print("Buscando todas las cliques maximales con Bron-Kerbosch v2...")
    cliques = bron_kerbosch_version2c(G)
    max_size = max(len(c) for c in cliques) if cliques else 0
    print(f"Número de cliques maximales encontradas: {len(cliques)}")
    print(f"Tamaño de la clique máxima: {max_size}")
    print(f"Por lo tanto, A({n},{d}) = {max_size}")
    
    # Mostrar algunas cliques grandes como ejemplo
    print("\nEjemplos de cliques maximales (hasta 5):")
    for i, clique in enumerate(cliques[:5]):
        print(f"  Clique {i+1}: tamaño {len(clique)} -> {clique}")

Construyendo grafo de Hamming H(6,4)...


NameError: name 'build_hamming_graph' is not defined

Nuevo codigo

In [8]:
def bron_kerbosch_max_clique2(adj_mask: list[int], N: int) -> int:
    """
    Algoritmo de Bron-Kerbosch con pivote y coloreo greedy para encontrar
    el tamaño de la clique máxima en el grafo representado por adj_mask.

    Parámetros
    ----------
    adj_mask : lista de N enteros.
               adj_mask[i] es un bitset donde el bit j está activo
               si existe arista entre el vértice i y el vértice j.
    N        : número de vértices del grafo (debe ser <= 63 para eficiencia,
               aunque Python soporta enteros arbitrariamente grandes).

    Retorna
    -------
    Tamaño (número de vértices) de la clique máxima encontrada.

    Estrategia
    ----------
    Los conjuntos se representan como enteros (bitsets):
      - P (candidatos)  : vértices que aún pueden extender la clique actual.
      - X (excluidos)   : vértices ya procesados (garantizan maximalidad).
      - r_size          : tamaño de la clique en construcción (reemplaza al
                          conjunto R, ya que solo necesitamos su cardinalidad).

    Podas aplicadas:
      1. Tamaño:   si r_size + |P| <= max_size, esta rama no puede mejorar.
      2. Coloreo:  si r_size + colores_greedy(P) <= max_size, ídem.
      3. Pivote:   se elige el vértice u en P∪X con mayor |P ∩ N(u)|,
                   reduciendo los candidatos a explorar a P \ N(u).
    """

    # Máscara de N bits para evitar bits "fantasma" al aplicar complemento (~)
    # En Python los enteros son de precisión arbitraria y con signo,
    # por lo que ~x activa infinitos bits superiores si no se enmascara.
    full_mask = (1 << N) - 1

    max_clique_size = 0   # mejor resultado encontrado hasta ahora

    # ── Coloreo greedy ──────────────────────────────────────────────────────────
    def greedy_color_bound(candidates_mask: int) -> int:
        """
        Devuelve una cota superior del tamaño de clique dentro de `candidates_mask`
        mediante coloreo greedy por clases de color independientes.

        Lógica: el tamaño de la clique máxima <= número cromático del grafo.
        Se construyen clases de color (conjuntos independientes) de forma greedy:
        cada vértice se asigna a la primera clase que no tenga ningún vecino suyo.
        El número de clases necesarias es la cota.
        """
        color_classes: list[int] = []   # cada elemento es un bitset (clase de color)
        remaining = candidates_mask

        while remaining:
            # Tomar el vértice de menor índice aún sin colorear
            vertex_bit = remaining & -remaining
            vertex = vertex_bit.bit_length() - 1

            # Buscar la primera clase existente sin vecinos de `vertex`
            placed = False
            for idx, color_class in enumerate(color_classes):
                if not (color_class & adj_mask[vertex]):
                    # Ningún nodo de esta clase es vecino de vertex → asignar aquí
                    color_classes[idx] |= vertex_bit
                    placed = True
                    break

            if not placed:
                # Ninguna clase sirve → abrir una nueva
                color_classes.append(vertex_bit)

            remaining &= ~vertex_bit

        return len(color_classes)

    # ── Expansión recursiva ─────────────────────────────────────────────────────
    def expand(r_size: int, P: int, X: int) -> None:
        """
        Extiende la clique actual (de tamaño r_size) probando cada candidato en P.

        Parámetros
        ----------
        r_size : número de vértices en la clique que se está construyendo.
        P      : bitset de candidatos que pueden ampliar la clique.
        X      : bitset de excluidos (ya procesados en ramas anteriores).
        """
        nonlocal max_clique_size

        # ── Caso base: clique maximal ───────────────────────────────────────────
        if P == 0 and X == 0:
            if r_size > max_clique_size:
                max_clique_size = r_size
            return

        # ── Poda 1: tamaño ─────────────────────────────────────────────────────
        # Si incluso añadiendo todos los candidatos no superamos el máximo, podar.
        if r_size + P.bit_count() <= max_clique_size:
            return

        # ── Poda 2: coloreo greedy ─────────────────────────────────────────────
        # Cota más ajustada: número cromático de P es cota del tamaño de clique en P.
        if r_size + greedy_color_bound(P) <= max_clique_size:
            return

        # ── Elegir pivote u en P ∪ X ───────────────────────────────────────────
        # Criterio: maximizar |P ∩ N(u)| para minimizar los candidatos a explorar.
        # Cuantos más candidatos cubre el pivote, menos ramas se abren.
        union_PX = P | X
        best_pivot = -1
        best_coverage = -1

        temp = union_PX
        while temp:
            u_bit = temp & -temp
            u = u_bit.bit_length() - 1
            coverage = (P & adj_mask[u]).bit_count()
            if coverage > best_coverage:
                best_coverage = coverage
                best_pivot = u
            temp ^= u_bit   # eliminar u_bit de temp

        # Candidatos a explorar: vértices de P que NO son vecinos del pivote.
        # El pivote ya "cubre" sus vecinos, así que no hay que expandirlos desde aquí.
        candidates = P & (~adj_mask[best_pivot] & full_mask)

        # ── Ciclo de backtracking ───────────────────────────────────────────────
        while candidates:
            # Tomar el candidato de menor índice
            v_bit = candidates & -candidates
            v = v_bit.bit_length() - 1

            # Llamada recursiva: añadir v a la clique,
            # restringir P y X a los vecinos de v.
            expand(
                r_size + 1,
                P & adj_mask[v],
                X & adj_mask[v],
            )

            # Mover v de P a X: ya fue procesado en esta rama.
            P &= ~v_bit
            X |= v_bit

            # Actualizar candidates eliminando directamente v.
            # Es equivalente a recalcular P & ~adj_mask[best_pivot] porque
            # v ya no está en P, y el pivote no cambia en este ciclo.
            candidates &= ~v_bit

    # ── Llamada inicial ─────────────────────────────────────────────────────────
    # Al inicio: r_size = 0, P = todos los vértices, X = vacío.
    all_vertices = full_mask
    expand(r_size=0, P=all_vertices, X=0)

    return max_clique_size

<>:2: SyntaxWarning: invalid escape sequence '\ '
<>:2: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_25190/93218190.py:2: SyntaxWarning: invalid escape sequence '\ '
  """


In [7]:
def hamming_distance(x: int, y: int, n: int) -> int:
    """
    Calcula la distancia de Hamming entre dos enteros x e y,
    considerando solo los n bits menos significativos.
    """
    return (x ^ y).bit_count()  # Python 3.8+: bit_count() es más rápido que bin().count()

def build_adjacency_mask(n: int, d: int) -> list:
    """
    Construye una lista de máscaras de adyacencia para el grafo H(n,d).
    - Cada vértice se representa por un entero de 0 a 2^n - 1.
    - adj_mask[i] es un entero cuyo bit j está a 1 si el vértice i es adyacente a j (j ≠ i y d_H(i,j) ≥ d).
    - Usamos un solo entero por vértice (Python int de precisión arbitraria), capaz de manejar hasta 2^n bits.
    """
    N = 1 << n                     # N = 2^n, número de vértices
    adj_mask = [0] * N             # Inicializar lista de máscaras

    # Precalculamos todas las distancias? Podríamos, pero O(N^2) es inevitable para construir el grafo.
    # Sin embargo, podemos acelerar usando la propiedad de que la distancia de Hamming es el número de unos en XOR.
    for i in range(N):
        # Para cada i, recorremos j > i y llenamos simétricamente
        for j in range(i + 1, N):
            if hamming_distance(i, j, n) >= d:
                # Establecer el bit j en la máscara de i
                adj_mask[i] |= (1 << j)
                # Y el bit i en la máscara de j
                adj_mask[j] |= (1 << i)
    return adj_mask, N

In [14]:
n= 9
d= 4
adj_mask, N = build_adjacency_mask(n,d)
result = bron_kerbosch_max_clique2(adj_mask, N)

/tmp/ipykernel_25190/93218190.py:2: SyntaxWarning: invalid escape sequence '\ '
  """


KeyboardInterrupt: 

In [13]:
result

16